<a href="https://colab.research.google.com/github/shin584/project/blob/3D_simulation/DNABERT_base_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ==========================================
# [추가] 완벽한 재현성을 위한 난수 및 CuDNN 고정
# ==========================================
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ==========================================
# Phase 1: 데이터 로드 및 전처리
# ==========================================
def load_and_preprocess(file_path):
    print(f"Loading {file_path}...")
    df = pd.read_csv(file_path)

    seq_col = next((c for c in df.columns if c.lower() in ["input_sequence", "sequence", "target_sequence", "seq"]), df.columns[0])
    label_col = next((c for c in df.columns if c.lower() in ["score", "label", "efficiency", "cleavage_efficiency", "cleavage_score"]), df.columns[1])

    sequences = df[seq_col].astype(str).tolist()
    labels = df[label_col].astype(float).tolist()

    # [추가] 서열 길이 동일성 검증 (CNN/LSTM 구조적 안정성 확보)
    seq_lengths = set(len(seq) for seq in sequences)
    if len(seq_lengths) > 1:
        raise ValueError(f"오류: 모든 서열의 길이가 동일해야 합니다. 발견된 길이들: {seq_lengths}")

    def seq_to_onehot(seq):
        mapping = {'A': [1,0,0,0], 'C': [0,1,0,0], 'G': [0,0,1,0], 'T': [0,0,0,1]}
        encoded = [mapping.get(nuc.upper(), [0,0,0,0]) for nuc in seq]
        return np.array(encoded).T

    X = torch.tensor(np.array([seq_to_onehot(seq) for seq in sequences]), dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

    return TensorDataset(X, y)

# ==========================================
# Phase 2: 순정 아키텍처 설계 (Vanilla Baseline)
# ==========================================
class VanillaBaseline(nn.Module):
    def __init__(self, in_channels=4, conv_out=128, lstm_hidden=64):
        super(VanillaBaseline, self).__init__()
        self.conv = nn.Conv1d(in_channels=in_channels, out_channels=conv_out, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(input_size=conv_out, hidden_size=lstm_hidden,
                            batch_first=True, bidirectional=True)
        self.fc = nn.Linear(lstm_hidden * 2, 1)

    def forward(self, x):
        x = self.relu(self.conv(x))
        x = x.permute(0, 2, 1)
        lstm_out, (hn, cn) = self.lstm(x)

        h_n_forward = hn[-2, :, :]
        h_n_backward = hn[-1, :, :]
        h_cat = torch.cat((h_n_forward, h_n_backward), dim=1)

        out = self.fc(h_cat)
        return out

# ==========================================
# Phase 3 & 4: 통제된 학습 루프 및 단일 평가
# ==========================================
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}\n")

    # [추가] DataLoader 셔플링 순서 고정용 제너레이터
    g = torch.Generator()
    g.manual_seed(SEED)

    batch_size = 64
    train_loader = DataLoader(load_and_preprocess("train.csv"), batch_size=batch_size, shuffle=True, generator=g)
    val_loader = DataLoader(load_and_preprocess("val.csv"), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(load_and_preprocess("test.csv"), batch_size=batch_size, shuffle=False)

    model = VanillaBaseline().to(device)
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-3)

    patience = 7
    best_val_loss = float('inf')
    patience_counter = 0
    save_path = "best_vanilla_baseline.pth"
    max_epochs = 100

    # [추가] Early Stopping min_delta 설정
    min_delta = 1e-4

    print("\n=== 시작: 순정 베이스라인 학습 ===")
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X_batch.size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * X_batch.size(0)

        val_loss /= len(val_loader.dataset)
        print(f"Epoch {epoch+1:03d}/{max_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # [수정] min_delta를 반영한 조기 종료 로직
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print("  --> Best Model Saved!")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n[Early Stopping] {patience} 에포크 동안 검증 손실의 유의미한 개선(>{min_delta})이 없어 조기 종료합니다.")
                break

    print("\n=== 최종 평가: Test Set 추론 ===")

    # [수정] CPU/GPU 환경 차이 방지를 위한 map_location 명시
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            all_preds.extend(outputs.cpu().numpy().flatten())
            all_labels.extend(y_batch.numpy().flatten())

    mae = mean_absolute_error(all_labels, all_preds)
    mse = mean_squared_error(all_labels, all_preds)

    # [수정] 모델이 동일한 값만 반환하여 분산이 0이 될 때의 NaN 반환 방어
    if np.std(all_preds) == 0 or np.std(all_labels) == 0:
        print("경고: 예측값 또는 실제값의 분산이 0입니다. 상관계수를 계산할 수 없어 0.0으로 처리합니다.")
        spearman_rho = 0.0
        pearson_r = 0.0
    else:
        spearman_rho, _ = spearmanr(all_labels, all_preds)
        pearson_r, _ = pearsonr(all_labels, all_preds)

    print("\n[순정 베이스라인 최종 성능 지표 (Floor)]")
    print("-" * 40)
    print(f"Spearman correlation (ρ) : {spearman_rho:.4f} (비교 핵심 지표)")
    print(f"Pearson correlation (r)  : {pearson_r:.4f}")
    print(f"Mean Absolute Error (MAE): {mae:.4f}")
    print(f"Mean Squared Error (MSE) : {mse:.4f}")
    print("-" * 40)

if __name__ == "__main__":
    main()

Using device: cpu

Loading train.csv...
Loading val.csv...
Loading test.csv...

=== 시작: 순정 베이스라인 학습 ===
Epoch 001/100 | Train Loss: 0.0626 | Val Loss: 0.0487
  --> Best Model Saved!
Epoch 002/100 | Train Loss: 0.0471 | Val Loss: 0.0458
  --> Best Model Saved!
Epoch 003/100 | Train Loss: 0.0426 | Val Loss: 0.0442
  --> Best Model Saved!
Epoch 004/100 | Train Loss: 0.0393 | Val Loss: 0.0369
  --> Best Model Saved!
Epoch 005/100 | Train Loss: 0.0363 | Val Loss: 0.0355
  --> Best Model Saved!
Epoch 006/100 | Train Loss: 0.0336 | Val Loss: 0.0339
  --> Best Model Saved!
Epoch 007/100 | Train Loss: 0.0326 | Val Loss: 0.0329
  --> Best Model Saved!
Epoch 008/100 | Train Loss: 0.0315 | Val Loss: 0.0324
  --> Best Model Saved!
Epoch 009/100 | Train Loss: 0.0314 | Val Loss: 0.0325
Epoch 010/100 | Train Loss: 0.0304 | Val Loss: 0.0315
  --> Best Model Saved!
Epoch 011/100 | Train Loss: 0.0293 | Val Loss: 0.0362
Epoch 012/100 | Train Loss: 0.0296 | Val Loss: 0.0312
  --> Best Model Saved!
Epoch 01